In [1]:
import sys
print(sys.executable)

import torch

print("PyTorch:",torch.__version__)

print("CUDA:",torch.cuda.is_available())

c:\Users\ASUS\anaconda3\envs\pytorch-learning\python.exe
PyTorch: 2.14.0+cpu
CUDA: False


In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from beams import ss_center

print("All libraries imported successfully!")

print("PyTorch version:",torch.__version__)
print("Training device:","cuda" if torch.cuda.is_available() else "cpu")

# Load training and validation datasets
train_df=pd.read_csv("train.csv")
val_df=pd.read_csv("val.csv")

print("\nTraining shape:",train_df.shape)
print("Validation shape:",val_df.shape)

print("\nDataset loaded successfully!")

All libraries imported successfully!
PyTorch version: 2.14.0+cpu
Training device: cpu

Training shape: (7350, 7)
Validation shape: (1575, 7)

Dataset loaded successfully!


In [4]:
# Select input and output columns
input_cols=["x","L","E","I","P"]
target_col="y"

# Convert data into NumPy arrays
X_train=train_df[input_cols].values
y_train=train_df[target_col].values.reshape(-1,1)

X_val=val_df[input_cols].values
y_val=val_df[target_col].values.reshape(-1,1)

# Convert NumPy arrays into PyTorch tensors
X_train=torch.tensor(X_train,dtype=torch.float32)
y_train=torch.tensor(y_train,dtype=torch.float32)

X_val=torch.tensor(X_val,dtype=torch.float32)
y_val=torch.tensor(y_val,dtype=torch.float32)

print("Training input shape:",X_train.shape)
print("Training output shape:",y_train.shape)

print("Validation input shape:",X_val.shape)
print("Validation output shape:",y_val.shape)

Training input shape: torch.Size([7350, 5])
Training output shape: torch.Size([7350, 1])
Validation input shape: torch.Size([1575, 5])
Validation output shape: torch.Size([1575, 1])


In [5]:
class BeamMLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.network=nn.Sequential(
            nn.Linear(5,64),
            nn.ReLU(),

            nn.Linear(64,64),
            nn.ReLU(),

            nn.Linear(64,1)
        )

    def forward(self,x):
        return self.network(x)


# Create the model
model=BeamMLP()

print(model)

BeamMLP(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [6]:
total_params=sum(p.numel() for p in model.parameters())

print("Total trainable parameters:",total_params)

Total trainable parameters: 4609


In [7]:
# Define loss function
criterion=nn.MSELoss()

# Define optimizer
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

print("Loss function:",criterion)
print("Optimizer:",optimizer)

Loss function: MSELoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [8]:

from torch.utils.data import TensorDataset,DataLoader
import copy

# Select training device
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

model=model.to(device)

# Create training batches
train_dataset=TensorDataset(X_train,y_train)

train_loader=DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

# Validation data
X_val_device=X_val.to(device)
y_val_device=y_val.to(device)

# Training settings
epochs=500
best_val_loss=float("inf")
best_epoch=0

# Store loss history
train_losses=[]
val_losses=[]

# Reinitialize optimizer
criterion=nn.MSELoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

print("Starting training...")
print("Device:",device)

for epoch in range(epochs):

    # Training phase
    model.train()
    running_loss=0.0

    for X_batch,y_batch in train_loader:

        X_batch=X_batch.to(device)
        y_batch=y_batch.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward propagation
        predictions=model(X_batch)

        # Calculate loss
        loss=criterion(predictions,y_batch)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss+=loss.item()*X_batch.size(0)

    epoch_train_loss=running_loss/len(train_dataset)

    # Validation phase
    model.eval()
    val_loss=0.0

    with torch.no_grad():

        predictions=model(X_val_device)

        loss=criterion(predictions,y_val_device)

        val_loss=loss.item()

    # Store losses
    train_losses.append(epoch_train_loss)
    val_losses.append(val_loss)

    # Save best model
    if val_loss<best_val_loss:

        best_val_loss=val_loss
        best_epoch=epoch+1

        torch.save(model.state_dict(),"best_model.pth")

    # Print losses for every epoch
    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {epoch_train_loss:.8f} "
        f"Val Loss: {val_loss:.8f}"
    )

print("\nTraining completed!")
print(f"Best validation loss: {best_val_loss:.8f}")
print(f"Best epoch: {best_epoch}")
print("Best model saved as best_model.pth")


Starting training...
Device: cpu
Epoch [1/500] Train Loss: 0.44421682 Val Loss: 0.14058739
Epoch [2/500] Train Loss: 0.06818609 Val Loss: 0.03136346
Epoch [3/500] Train Loss: 0.02578003 Val Loss: 0.01825360
Epoch [4/500] Train Loss: 0.01673840 Val Loss: 0.01362206
Epoch [5/500] Train Loss: 0.01223072 Val Loss: 0.00994089
Epoch [6/500] Train Loss: 0.00923940 Val Loss: 0.00893071
Epoch [7/500] Train Loss: 0.00733791 Val Loss: 0.00678610
Epoch [8/500] Train Loss: 0.00558860 Val Loss: 0.00547749
Epoch [9/500] Train Loss: 0.00462396 Val Loss: 0.00445125
Epoch [10/500] Train Loss: 0.00395552 Val Loss: 0.00415590
Epoch [11/500] Train Loss: 0.00352092 Val Loss: 0.00335387
Epoch [12/500] Train Loss: 0.00290094 Val Loss: 0.00281278
Epoch [13/500] Train Loss: 0.00242677 Val Loss: 0.00287554
Epoch [14/500] Train Loss: 0.00223476 Val Loss: 0.00249766
Epoch [15/500] Train Loss: 0.00185737 Val Loss: 0.00211740
Epoch [16/500] Train Loss: 0.00175976 Val Loss: 0.00198479
Epoch [17/500] Train Loss: 0.001

In [ ]:
import matplotlib as plt
plt.figure(figsize=(9,5))

plt.plot(train_losses,label="Training Loss")
plt.plot(val_losses,label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training and Validation Loss")

plt.yscale("log")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("learning_curve.png",dpi=300)
plt.show()

NameError: name 'plt' is not defined